In [0]:
from pyspark.sql import functions as F

# Load the table
df = spark.table("post_renewal_churn.raw.billings_cleaned")

# Apply the transformation: change 'Won' to 'Churned' when conditions are met
df_updated = df.withColumn(
    "prospect_outcome",
    F.when(
        (F.col("Closed_Date") >= F.col("Prospect_Renewal_Date")) &
        (F.datediff(F.col("Closed_Date"), F.col("Prospect_Renewal_Date")) >= 30) &
        (F.col("prospect_outcome") == "Won"),
        "Churned"
    ).otherwise(F.col("prospect_outcome"))
)

# Count the records that were changed
count_changed = df_updated.filter(
    (F.col("Closed_Date") >= F.col("Prospect_Renewal_Date")) &
    (F.datediff(F.col("Closed_Date"), F.col("Prospect_Renewal_Date")) >= 30) &
    (F.col("prospect_outcome") == "Churned")
)


print(f"Records changed from 'Won' to 'Churned': {count_changed}")

# Display the updated DataFrame
display(df_updated)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ── 1. Load / prepare billings ──────────────────────────────────────────
df_billings = spark.table("post_renewal_churn.raw.billings_cleaned") \
    .withColumn("index", F.monotonically_increasing_id())


df_calls = spark.table("post_renewal_churn.raw.renewal_call_cleaned")


df_calls = df_calls \
    .filter(F.col("Analysed_Call") == "1")

# ── 3. Join on co_ref — explicit condition to avoid ambiguous reference ──
df_joined = df_billings.join(
    df_calls,
    on=df_billings["Co_Ref"] == df_calls["Co_Ref"],
    how="left"
).filter(
    (F.col("Call_Date") >= F.col("prospect_renewal_date")) &
    (F.col("Call_Date") <= F.col("closed_date"))
).drop(df_calls["Co_Ref"])

# ── 4. Keep only the LATEST call per billing row (index) ────────────────
window = Window.partitionBy("index").orderBy(F.desc("Call_Date"))

df_latest_call = df_joined \
    .withColumn("rn", F.row_number().over(window)) \
    .filter(F.col("rn") == 1) \
    .drop("rn")

df_latest_call.display()
df_final = df_latest_call
df_final.write.mode("overwrite").saveAsTable("post_renewal_churn.cleaned_dataset.final_dataset")

In [0]:
df_final.groupBy("prospect_outcome").count().display()

In [0]:
count_changed = df_final.filter(
    (F.col("Closed_Date") >= F.col("Prospect_Renewal_Date")) &
    (F.datediff(F.col("Closed_Date"), F.col("Prospect_Renewal_Date")) >= 30) &
    (F.col("prospect_outcome") == "Churned")
).count()
print(count_changed)


In [0]:
display(df_final)